In [4]:
import pandas as pd
from pathlib import Path

data_path = Path(r"D:\Documents\FLIGHTOPS\data\raw")

print("Pandas Version:", pd.__version__)
print("Raw Data folder exists:", data_path.exists())

Pandas Version: 3.0.0
Raw Data folder exists: True


In [6]:
csv_files = list(data_path.glob("*/*.csv"))

print("Total CSV files:- ", len(csv_files))

for file in csv_files:
    print(file)

Total CSV files:-  12
D:\Documents\FLIGHTOPS\data\raw\april2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\AUG2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\dec2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\Feburary2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\January2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\July2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\June2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\march2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\may2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\nov2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\oct2025\T_ONTIME_REPORTING.csv
D:\Documents\FLIGHTOPS\data\raw\sept2025\T_ONTIME_REPORTING.csv


In [8]:
for files in csv_files:
    df = pd.read_csv(file, nrows=5)
    print(f"\nFile: {file.parent.name}")
    print("Columns: ", df.columns.tolist())


File: sept2025
Columns:  ['MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY_NEW', 'DEP_DEL15', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED']

File: sept2025
Columns:  ['MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY_NEW', 'DEP_DEL15', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED']

File: sept2025
Columns:  ['MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY_NEW', 'DEP_DEL15', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED']

File: sept2025
Columns:  ['MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY_NEW', 'DEP_DEL15', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED']

File: sept2025
Columns:  ['MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'DEP_DELAY_NEW', 'D

In [9]:

column_sets = {}

for file in csv_files:
    df = pd.read_csv(file, nrows=5)
    month = file.parent.name
    column_sets[month] = df.columns.tolist()

    print(month, "->", len(df.columns), "columns")

first_month = next(iter(column_sets))
expected_columns = column_sets[first_month]

all_same = all(
    columns == expected_columns
    for columns in column_sets.values()
)

print("\nAll files have same columns:", all_same)


april2025 -> 14 columns
AUG2025 -> 14 columns
dec2025 -> 14 columns
Feburary2025 -> 14 columns
January2025 -> 14 columns
July2025 -> 14 columns
June2025 -> 14 columns
march2025 -> 14 columns
may2025 -> 14 columns
nov2025 -> 14 columns
oct2025 -> 14 columns
sept2025 -> 14 columns

All files have same columns: True


In [16]:

all_months = []

csv_files = sorted(
    csv_files,
    key=lambda file: int(file.parent.name.replace("2025", "").replace("January", "1").replace("Feburary", "2").replace("march", "3").replace("april", "4").replace("may", "5").replace("June", "6").replace("July", "7").replace("AUG", "8").replace("sept", "9").replace("oct", "10").replace("nov", "11").replace("dec", "12"))
)

for file in csv_files:
    df = pd.read_csv(file)
    all_months.append(df)

flights = pd.concat(all_months, ignore_index=True)

print(flights["MONTH"].head())


0    1
1    1
2    1
3    1
4    1
Name: MONTH, dtype: int64


In [17]:
print(sorted(flights["MONTH"].unique()))

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


In [18]:

flights.to_csv(
    r"D:\Documents\FLIGHTOPS\data\processed\flights_2025.csv",
    index=False
)

print("Master dataset saved successfully!")


Master dataset saved successfully!


## STEP 2: CHECKING MISSING VALUES IN THE FINAL DATASET

### Objective
To identify the number of missing values in each column of the combined flight dataset.

In [21]:
missing_values = flights.isnull().sum()

print(missing_values)

MONTH                      0
DAY_OF_WEEK                0
FL_DATE                    0
OP_UNIQUE_CARRIER          0
ORIGIN                     0
DEST                       0
DEP_DELAY_NEW          98591
DEP_DEL15              98591
ARR_DELAY             122135
ARR_DELAY_NEW         122135
ARR_DEL15             122135
CANCELLED                  0
CANCELLATION_CODE    6898743
DIVERTED                   0
dtype: int64



## STEP 3: CALCULATING MISSING VALUE PERCENTAGES

### Objective
To calculate the percentage of missing values in each column of the final dataset.

### Why Are We Doing This?
Missing value counts alone do not show how much of a column's data is missing. Percentages help us understand the extent of missing data and decide which columns need further investigation.

### Expected Outcome
The missing value percentage for each column in the dataset.

### Important Note
No missing values will be deleted or replaced at this stage. We will investigate the reasons for missing values before deciding how to handle them.


In [22]:
missing_percent = (flights.isnull().sum() / len(flights)) * 100

print(missing_percent.round(2))

MONTH                 0.00
DAY_OF_WEEK           0.00
FL_DATE               0.00
OP_UNIQUE_CARRIER     0.00
ORIGIN                0.00
DEST                  0.00
DEP_DELAY_NEW         1.41
DEP_DEL15             1.41
ARR_DELAY             1.74
ARR_DELAY_NEW         1.74
ARR_DEL15             1.74
CANCELLED             0.00
CANCELLATION_CODE    98.53
DIVERTED              0.00
dtype: float64


In [23]:
print(flights["CANCELLED"].value_counts())

CANCELLED
0.0    6898743
1.0     102876
Name: count, dtype: int64


## Next step: doing Actual relationship verification

In [24]:
print(pd.crosstab(
    flights["CANCELLED"],
    flights["CANCELLATION_CODE"].isnull()
))

CANCELLATION_CODE   False    True 
CANCELLED                         
0.0                     0  6898743
1.0                102876        0


In [25]:
print(pd.crosstab(
    flights["CANCELLED"],
    flights["ARR_DELAY"].isnull()
))

ARR_DELAY    False   True 
CANCELLED                 
0.0        6879484   19259
1.0              0  102876


## Next step: Checking  whether diverted flights explain the remaining missing values

In [26]:
print(pd.crosstab(
    flights["DIVERTED"],
    flights["ARR_DELAY"].isnull()
))

ARR_DELAY    False   True 
DIVERTED                  
0.0        6879484  102877
1.0              0   19258


## Next step: Finding the remaining missing arrival delay

In [28]:
missing_reason = flights[
    (flights["ARR_DELAY"].isnull()) &
    (flights["CANCELLED"] == 0) &
    (flights["DIVERTED"] == 0)
]

print("Remaining records:", len(missing_reason))
print(missing_reason)

Remaining records: 1
         MONTH  DAY_OF_WEEK               FL_DATE OP_UNIQUE_CARRIER ORIGIN  \
5112899      9            6  9/6/2025 12:00:00 AM                YX    JFK   

        DEST  DEP_DELAY_NEW  DEP_DEL15  ARR_DELAY  ARR_DELAY_NEW  ARR_DEL15  \
5112899  MVY            0.0        0.0        NaN            NaN        NaN   

         CANCELLED CANCELLATION_CODE  DIVERTED  
5112899        0.0               NaN       0.0  


In [ ]:
print(pd.crosstab(
    flights["CANCELLED"],
    flights["DEP_DELAY_NEW"].isnull()
))

## Next task: Checking departure delay against diverted flights

In [30]:
print(pd.crosstab(
    flights["DIVERTED"],
    flights["DEP_DELAY_NEW"].isnull()
))

DEP_DELAY_NEW    False  True 
DIVERTED                     
0.0            6883770  98591
1.0              19258      0


## Next task: Verifying the exact relationship

In [31]:
print(pd.crosstab(
    flights["CANCELLED"],
    flights["DEP_DELAY_NEW"].isnull()
))

DEP_DELAY_NEW    False  True 
CANCELLED                    
0.0            6898743      0
1.0               4285  98591
